In [1]:
import pandas as pd
df = pd.read_csv("data/commodity_prices.csv")
df.head()

,State,District,Market,Commodity,Variety,Grade,Arrival_Date,Min_Price,Max_Price,Modal_Price,Commodity_Code
0,Kerala,Ernakulam,Aluva,Potato,Potato,FAQ,06/07/2024,4000.0,5000,4500,24
1,Kerala,Ernakulam,Aluva,Potato,Potato,FAQ,11/07/2024,4000.0,5000,4500,24
2,Kerala,Ernakulam,Aluva,Potato,Potato,FAQ,17/07/2024,4000.0,5000,4500,24
3,Kerala,Ernakulam,Aluva,Ginger(Green),Green Ginger,FAQ,03/07/2024,14000.0,18000,16000,103
4,Kerala,Ernakulam,Aluva,Ginger(Green),Green Ginger,FAQ,04/07/2024,14000.0,18000,16000,103


In [2]:
from utils.wrangle_model_data import wrangle_ml

df1 = wrangle_ml(df)
df1.head()

,Arrival_Date,Product_Type,Market,Modal_Price,log_Modal_Price_filled,lag_1,lag_3,lag_7,lag_14,lag_30,...,rolling_std_3,rolling_mean_7,rolling_std_7,rolling_mean_14,rolling_std_14,Commodity,Variety_Type,Is_VFPCK,Season,Year
0,2023-12-13,Alsandikai|Alsandikai|FAQ,North Paravur,5200.0,8.556606,NaN,NaN,NaN,NaN,NaN,...,NaN,8.556606,NaN,8.556606,NaN,Alsandikai,Alsandikai|Alsandikai,False,Winter,2023
1,2023-12-14,Alsandikai|Alsandikai|FAQ,North Paravur,6200.0,8.732466,8.556606,NaN,NaN,NaN,NaN,...,0.124352,8.644536,0.124352,8.644536,0.124352,Alsandikai,Alsandikai|Alsandikai,False,Winter,2023
2,2023-12-15,Alsandikai|Alsandikai|FAQ,North Paravur,NaN,8.732466,8.732466,NaN,NaN,NaN,NaN,...,0.101533,8.673846,0.101533,8.673846,0.101533,Alsandikai,Alsandikai|Alsandikai,False,Winter,2023
3,2023-12-16,Alsandikai|Alsandikai|FAQ,North Paravur,4800.0,8.476580,8.732466,8.556606,NaN,NaN,NaN,...,0.147736,8.624529,0.128845,8.624529,0.128845,Alsandikai,Alsandikai|Alsandikai,False,Winter,2023
4,2023-12-17,Alsandikai|Alsandikai|FAQ,North Paravur,NaN,8.476580,8.476580,8.732466,NaN,NaN,NaN,...,0.147736,8.594939,0.129725,8.594939,0.129725,Alsandikai,Alsandikai|Alsandikai,False,Winter,2023


Step A: Visual storytelling
	•	Create the bar chart + heatmap.
	•	From this, identify the 2–3 largest groups.

Step B: Case studies
	•	Take one product type from each large group.
	•	Plot its modal price time series.
	•	Annotate with Market, Season, etc. to see if the patterns align with your eta2 findings.

Step C: Link to forecasting
	•	For each case study, brainstorm: “If I were to predict this, which features should my model use?”
	•	Write down hypotheses:
	•	“Banana in Ernakulam → strongly seasonal, market doesn’t matter much”
	•	“Tomato → huge market-driven swings, must include Market feature”

This will connect exploration → modeling plan.

How to pick the example per group

1) Coverage (enough data)

Prefer rows with larger Total_Records (use log to avoid letting whales dominate).

2) Dominance (right signal for that group)

For each row, let D = features listed in important_features and O = the rest.

Compute:
	•	signal = geometric mean of the effect sizes for features in D
	•	noise = arithmetic mean of effect sizes for features in O
	•	dominance_score = signal / (noise + ε)
	•	score = dominance_score × log(1 + Total_Records)

Pick the top score row as your “high-signal” exemplar.

3) Representativeness (typical, not extreme)

Within each group, also compute the L1 distance of each row’s effect-size vector to the group medians across all five effects. Pick the smallest distance row as your “representative” exemplar.

Result: for each important_features group you’ll have two great candidates:
(A) the most clear-cut, high-signal case; (B) the most typical case. If you want only one, default to (B) unless you’re building intuition; then use (A).

4) Stability sanity check (eta² vs ω²)

You already flagged inflated η². If you have ω² at the same grain, filter out rows where any dominant feature has:
	•	eta2 - omega2 > 0.10 or eta2/omega2 > 1.5.
This avoids picking an example whose “importance” is just small-sample inflation.

3 — Product & Market identity / static context

Why: heterogeneity across markets/products is big (price levels, volatility). Use it directly.

What to use
	•	Product_Type, Market, Variety_Type, Is_VFPCK, Season, Year (keep Year as numeric or category)
	•	Your precomputed effect-size features: Mean_Commodity_Effect_Size, Mean_Variety_Type_Effect_Size, Mean_Season_Effect_Size, Mean_Market_Effect_Size, Mean_Year_Effect_Size, Total_Records

6 — Aggregation / target-encoding-like features (group-level statistics)

Why: encode long-term group behavior (typical price levels, historical volatility) so trees can compare current value to long-term baselines.

Useful features
	•	global_product_mean — mean price for Product_Type (across markets) in training period
	•	market_mean — mean price for Market (across products)
	•	product_market_mean — mean price for Product_Type × Market
	•	expanding_mean per group: cumulative mean up to day t-1 (no leakage)
	•	expanding_std per group

How to compute safely (no leakage):

9 — Interaction features

Why: some effects only appear jointly (e.g., Season matters differently across Markets).

Examples
	•	Market_Season = Market + '|' + Season (as category)
	•	Product_Month = Product_Type + '_' + month
	•	Cross features of Is_VFPCK × Season